# Session 2: Probability and Bayesian Updating
### Conditional Probability, Natural Frequencies & Sequential Learning
*Bayesian Cognitive Science & Data Analysis (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/02_probability_bayesian_updating.ipynb)

---

## 1. Setup & Environment Initialization
This cell detects whether you are running locally or in Google Colab, configures Plotly inline rendering, and downloads course datasets from GitHub if needed.


In [ ]:
# ==============================================================================
# 🚀 1. Setup Cell: Auto-Detect Environment & Fetch Course Data
# ==============================================================================
import sys
import os
import urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

# Auto-fetch UCB Admissions dataset
DATA_DIR = "data"
DATA_FILE = "ucbadmit.csv"
LOCAL_PATH = os.path.join(DATA_DIR, DATA_FILE)
RAW_URL = f"https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/{DATA_FILE}"

if not os.path.exists(LOCAL_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📥 Downloading {DATA_FILE} from GitHub...")
    urllib.request.urlretrieve(RAW_URL, LOCAL_PATH)
    print(f"✅ Downloaded to {LOCAL_PATH}")
else:
    print(f"📂 Found existing local dataset at {LOCAL_PATH}")

df_ucb = pd.read_csv(LOCAL_PATH)
print(f"📊 Berkeley Admissions table loaded: {len(df_ucb)} rows.")
df_ucb.head()


## 2. Conditional Probability & Denominators: Berkeley Admissions (1973)
Bickel, Hammel, and O'Connell (1975, *Science*) analyzed 4,526 graduate applications across six major departments.

In Bayesian conditioning, **the condition defines the denominator**. Changing the condition changes which subset of the world you are analyzing.


In [ ]:
# Verify overall aggregate totals
total_apps = df_ucb['applications'].sum()
total_admits = df_ucb['admit'].sum()
print(f"Total Applications: {total_apps}, Total Admits: {total_admits}")

# Overall admissions by gender
by_gender = df_ucb.groupby('gender')[['admit', 'reject', 'applications']].sum()
by_gender['p_admit_given_gender'] = by_gender['admit'] / by_gender['applications']
print("\n--- Overall Rates ---")
print(by_gender[['admit', 'applications', 'p_admit_given_gender']])

# Reversing the condition: P(Female | Admitted)
female_admits = df_ucb[df_ucb['gender'] == 'female']['admit'].sum()
p_female_given_admit = female_admits / total_admits
print(f"\nP(Female | Admitted) = {female_admits}/{total_admits} = {p_female_given_admit:.4f}")

# Conditioning within Department A
dept_a = df_ucb[df_ucb['dept'] == 'A'].copy()
dept_a['p_admit'] = dept_a['admit'] / dept_a['applications']
print("\n--- Department A ---")
print(dept_a[['dept', 'gender', 'admit', 'applications', 'p_admit']])


## 3. The False Positive Paradox & Natural Frequencies
When screening for a condition with low base rate, the false positives from the healthy majority easily outnumber the true positives from the diseased minority.

Let us define a general Bayesian update function for binary tests:
$$
\mathbb{P}(C \mid +) = rac{	ext{Sens} 	imes 	ext{Prev}}{	ext{Sens} 	imes 	ext{Prev} + (1 - 	ext{Spec}) 	imes (1 - 	ext{Prev})}
$$


In [ ]:
def bayesian_screening(prevalence, sensitivity=0.95, false_positive_rate=0.05):
    p_pos_given_c = sensitivity
    p_pos_given_not_c = false_positive_rate
    
    numerator = p_pos_given_c * prevalence
    denominator = numerator + p_pos_given_not_c * (1 - prevalence)
    return numerator / denominator

# Compare general population screening vs specialist clinic
p_general = bayesian_screening(prevalence=0.005) # 0.5% base rate
p_clinic = bayesian_screening(prevalence=0.100)  # 10% base rate

print(f"General Screening (0.5% prevalence): P(C | +) = {p_general * 100:.2f}%")
print(f"High-Risk Clinic  (10.0% prevalence): P(C | +) = {p_clinic * 100:.2f}%")


In [ ]:
# Generate posterior probability curve across full spectrum of base rates
prev_grid = np.linspace(0.0001, 0.30, 300)
post_grid = [bayesian_screening(p, sensitivity=0.95, false_positive_rate=0.05) * 100 for p in prev_grid]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prev_grid * 100, y=post_grid, mode='lines',
    line=dict(color='#2b6cb0', width=2.5),
    name='Posterior P(C | +)'
))
fig.add_trace(go.Scatter(
    x=[0.5, 10.0], y=[p_general * 100, p_clinic * 100], mode='markers+text',
    text=['Screening (8.7%)', 'Clinic (67.9%)'], textposition=['top left', 'bottom right'],
    marker=dict(size=10, color=['#c53030', '#276749']),
    showlegend=False
))

fig.update_layout(
    title='Posterior Probability as a Function of Prior Prevalence',
    xaxis_title='Prior Base Rate Prevalence (%)',
    yaxis_title='Posterior P(C | +) (%)',
    template='plotly_white',
    height=420
)
fig.show()


## 4. Sequential Bernoulli Updating & Conjugacy
In sequential Bayesian updating, **yesterday's posterior becomes today's prior**:
$$
\operatorname{Beta}(lpha, eta) + y_t \implies \operatorname{Beta}(lpha + y_t, eta + 1 - y_t)
$$
Let us simulate the exact sequence of 5 coin flips from Chapter 2: $[1, 0, 1, 1, 0]$.


In [ ]:
# Sequential flips: H, T, H, H, T
flips = [1, 0, 1, 1, 0]
alpha_0, beta_0 = 2, 2

history = [{'step': 0, 'flip': None, 'alpha': alpha_0, 'beta': beta_0, 'mean': alpha_0 / (alpha_0 + beta_0)}]

curr_a, curr_b = alpha_0, beta_0
for t, y in enumerate(flips, start=1):
    curr_a += y
    curr_b += (1 - y)
    history.append({
        'step': t,
        'flip': 'Heads' if y == 1 else 'Tails',
        'alpha': curr_a,
        'beta': curr_b,
        'mean': curr_a / (curr_a + curr_b)
    })

df_steps = pd.DataFrame(history)
print("Sequential Beta Updating History:")
print(df_steps.to_string(index=False))


## 5. ROS Election Survey: Precision-Weighted Information Aggregation
Gelman, Hill, and Vehtari (*Regression and Other Stories*, Ch. 9) demonstrate Bayesian updating via precision weighting:

1. **Model-based Prior**: Historical economic forecast $\mu_0 = 0.524$, $\sigma_0 = 0.041$.
2. **Survey Evidence**: $n = 400$ voters, $y = 190$ supporting Democrat $\implies ar{y} = 190/400 = 0.475$, $\sigma_{	ext{survey}} = \sqrt{0.475 	imes 0.525 / 400} pprox 0.02497$.
3. **Precision-Weighted Posterior**:
$$
	au_0 = rac{1}{\sigma_0^2}, \quad 	au_d = rac{1}{\sigma_{	ext{survey}}^2}, \quad 	au_{	ext{post}} = 	au_0 + 	au_d
$$
$$
\mu_{	ext{post}} = rac{	au_0 \mu_0 + 	au_d ar{y}}{	au_{	ext{post}}}, \quad \sigma_{	ext{post}} = rac{1}{\sqrt{	au_{	ext{post}}}}
$$


In [ ]:
mu_0 = 0.524
se_0 = 0.041

n_survey = 400
y_survey = 190
y_bar = y_survey / n_survey
se_survey = np.sqrt(y_bar * (1 - y_bar) / n_survey)

tau_0 = 1 / (se_0 ** 2)
tau_d = 1 / (se_survey ** 2)
tau_post = tau_0 + tau_d

mu_post = (tau_0 * mu_0 + tau_d * y_bar) / tau_post
se_post = 1 / np.sqrt(tau_post)

print(f"Prior:     Mean = {mu_0:.4f}, SE = {se_0:.4f} (Weight = {tau_0 / tau_post * 100:.1f}%)")
print(f"Survey:    Mean = {y_bar:.4f}, SE = {se_survey:.4f} (Weight = {tau_d / tau_post * 100:.1f}%)")
print(f"Posterior: Mean = {mu_post:.4f}, SE = {se_post:.4f}")
